# NB02 - Label Quality & Confident Learning
## Background
Label noise is pervasive in real datasets — crowdsourcing platforms have 5-30% error rates, automated labeling pipelines introduce systematic biases, and even expert annotations disagree. Confident Learning (Northcutt et al., 2021; the theory behind Cleanlab) provides a principled framework for identifying mislabeled examples without requiring a second human review.

The key insight: if a model assigns high probability to class j for a sample labeled as class i, that sample is likely mislabeled. By thresholding per-class confidence on out-of-sample predictions (cross-validation), confident learning constructs a noise transition matrix C[i][j] = count of samples labeled i that are actually class j.

## What You'll Learn
- Confident learning: estimating the noise transition matrix
- Per-class quality thresholds from cross-validation probabilities
- Identifying mislabeled examples and their likely true class
- Quality score per sample: confidence-weighted agreement

## Why This Matters
Cleanlab is used by production ML teams at companies from Tesla to the UK NHS. Removing the top 10% noisiest samples often improves model performance more than adding 30% more clean data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict
from collections import Counter

# ── Generate dataset with known label noise ────────────────────────────────
np.random.seed(42)

def make_labeled_dataset(n_samples=800, n_classes=4, n_features=5,
                          noise_rate=0.15, seed=42):
    np.random.seed(seed)
    n_per_class = n_samples // n_classes
    class_means = np.eye(n_classes) * 3
    X, y_true = [], []
    for c in range(n_classes):
        feats = class_means[c, :n_features] + np.random.randn(n_per_class, n_features)
        X.append(feats); y_true.extend([c] * n_per_class)
    X, y_true = np.vstack(X), np.array(y_true)

    # Apply label noise
    y_noisy = y_true.copy()
    n_noisy = int(len(y_true) * noise_rate)
    noisy_idx = np.random.choice(len(y_true), n_noisy, replace=False)
    # Confuse similar classes: 0->1, 1->0, 2->3, 3->2
    confused = {0: 1, 1: 0, 2: 3, 3: 2}
    for idx in noisy_idx:
        y_noisy[idx] = confused.get(y_true[idx], (y_true[idx]+1) % n_classes)

    return X, y_true, y_noisy, noisy_idx

X, y_true, y_noisy, true_noisy_idx = make_labeled_dataset()
n_classes = 4

# ── Simulate cross-validation probabilities (oracle model) ─────────────────
def simulate_cv_probs(X, y_true, y_noisy, n_classes, noise_sigma=0.1):
    """Simulate softmax probs that a good CV model would produce."""
    probs = np.zeros((len(X), n_classes))
    for i, (yt, yn) in enumerate(zip(y_true, y_noisy)):
        # Model predicts based on true class but with noise
        logits = np.random.randn(n_classes) * noise_sigma
        logits[yt] += 3.0  # Strong signal for true class
        logits -= logits.max()
        exp_l = np.exp(logits)
        probs[i] = exp_l / exp_l.sum()
    return probs

probs = simulate_cv_probs(X, y_true, y_noisy, n_classes)
print(f'Dataset: {len(X)} samples, {n_classes} classes')
print(f'True noise rate: {len(true_noisy_idx)/len(X)*100:.1f}%')
print(f'Mean confidence for correct label: {probs[np.arange(len(y_true)), y_true].mean():.3f}')

In [ ]:
# ── Confident Learning: noise transition matrix ────────────────────────────
def confident_learning(probs: np.ndarray, y_noisy: np.ndarray,
                        n_classes: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Estimate noise transition matrix and identify mislabeled samples.
    Returns: noise_matrix C[given][true], suspected_idx, quality_scores
    """
    n = len(y_noisy)

    # Per-class threshold: average probability of samples in each class
    thresholds = np.zeros(n_classes)
    for c in range(n_classes):
        mask = y_noisy == c
        if mask.sum() > 0:
            thresholds[c] = probs[mask, c].mean()

    # Confident joint: count samples where p[true] >= threshold[true] and label=given
    C = np.zeros((n_classes, n_classes), dtype=int)
    for i in range(n):
        given_class = y_noisy[i]
        for true_class in range(n_classes):
            if probs[i, true_class] >= thresholds[true_class]:
                C[given_class, true_class] += 1

    # Identify mislabeled: model confident about different class than label
    predicted_class = np.argmax(probs, axis=1)
    suspected_mask = (predicted_class != y_noisy) & (probs.max(axis=1) > 0.7)
    suspected_idx = np.where(suspected_mask)[0]

    # Quality score per sample: probability of given label
    quality_scores = probs[np.arange(n), y_noisy]

    return C, suspected_idx, quality_scores

C, suspected_idx, quality_scores = confident_learning(probs, y_noisy, n_classes)

# Evaluation
true_noisy_set = set(true_noisy_idx)
suspected_set  = set(suspected_idx)
precision = len(true_noisy_set & suspected_set) / max(len(suspected_set), 1)
recall    = len(true_noisy_set & suspected_set) / max(len(true_noisy_set), 1)

print('=== Confident Learning Results ===')
print(f'Suspected mislabeled: {len(suspected_idx)} samples')
print(f'True mislabeled: {len(true_noisy_idx)} samples')
print(f'Precision: {precision:.3f} (of suspected, fraction truly noisy)')
print(f'Recall:    {recall:.3f} (of truly noisy, fraction detected)')
print()
print('Noise transition matrix C[given][true]:')
print(C)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Noise matrix heatmap
im = axes[0].imshow(C, cmap='YlOrRd')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Confident Joint C[given][true]')
axes[0].set_xlabel('True class'); axes[0].set_ylabel('Given (noisy) class')
for i in range(n_classes):
    for j in range(n_classes):
        axes[0].text(j, i, str(C[i,j]), ha='center', va='center', fontsize=10)

# Quality score distribution
clean_scores = quality_scores[~np.isin(np.arange(len(y_noisy)), true_noisy_idx)]
noisy_scores = quality_scores[np.isin(np.arange(len(y_noisy)), true_noisy_idx)]
axes[1].hist(clean_scores, bins=30, alpha=0.7, label='Clean samples', color='green')
axes[1].hist(noisy_scores, bins=30, alpha=0.7, label='Noisy samples', color='red')
axes[1].set_title('Quality Scores: Clean vs Noisy')
axes[1].set_xlabel('P(given label)'); axes[1].set_ylabel('Count')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Sorted quality scores
sorted_q = np.sort(quality_scores)
axes[2].plot(sorted_q, linewidth=1.5)
axes[2].axhline(0.5, color='red', linestyle='--', label='Quality threshold = 0.5')
axes[2].set_title('Sorted Quality Scores')
axes[2].set_xlabel('Sample rank'); axes[2].set_ylabel('Quality score')
n_below = (quality_scores < 0.5).sum()
axes[2].text(0.6, 0.3, f'{n_below} samples\nbelow threshold',
             transform=axes[2].transAxes, fontsize=10, color='red')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s29_02_label_quality.png', dpi=100, bbox_inches='tight')
plt.show()